<a href="https://colab.research.google.com/github/marcocintra/Atmosphere/blob/master/Gopi_TEC_Manaus_analysis_09_27_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gopi TEC Manaus analysis - 09/27/2024

In [ ]:
#Download Gopi_TEC_NAUS_Dec_2024_and_09_27_2024.zip from https://doi.org/10.5281/zenodo.15453941

In [ ]:
%pwd

In [ ]:
!ls

In [ ]:
!unzip 'Gopi_TEC_NAUS_Dec_2024_and_09_27_2024.zip'

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

df = pd.read_csv('naus271-2024-09-27.Std', sep='\s+', header=None)

df.columns = ['time_ut', 'tec', 'tec_std', 'latitude']

base_date = datetime(2024, 9, 27)

def decimal_to_time(decimal_hours):
    hours = int(decimal_hours)
    minutes = int((decimal_hours - hours) * 60)
    seconds = int(((decimal_hours - hours) * 60 - minutes) * 60)
    return base_date + timedelta(hours=hours, minutes=minutes, seconds=seconds)

df['DATETIME'] = df['time_ut'].apply(decimal_to_time)

df['tec'] = pd.to_numeric(df['tec'].replace('-', np.nan))
df['tec_std'] = pd.to_numeric(df['tec_std'].replace('-', np.nan))

df = df[['DATETIME', 'tec', 'tec_std', 'latitude']]
df.columns = ['DATETIME', 'TEC', 'TEC_STD', 'LATITUDE']

print(df)

df.to_pickle('sjsp271-2024-09-27.pkl')

In [ ]:
df

In [ ]:
import pandas as pd
from datetime import time

df['DATETIME'] = pd.to_datetime(df['DATETIME'])

df['hour'] = df['DATETIME'].dt.hour
df['minute'] = df['DATETIME'].dt.minute

filtro = (
    # 0:50
    ((df['hour'] == 0) & (df['minute'] == 50)) |
    # 1:00
    ((df['hour'] == 1) & (df['minute'] == 0)) |
    # 1:10
    ((df['hour'] == 1) & (df['minute'] == 10))
)

df_filtrado = df[filtro]

df_filtrado = df_filtrado.drop(columns=['hour', 'minute'])

print("DataFrame filtered for times 0:50, 01:00 and 01:10:")
print(df_filtrado)

In [ ]:
reference_time = pd.Timestamp('2024-09-27 00:50:00')
wide_window = timedelta(minutes=5)
df_proximos = df[(df['DATETIME'] >= reference_time - wide_window) &
                 (df['DATETIME'] <= reference_time + wide_window)]
print("Records close to 00:50:")
print(df_proximos[['DATETIME', 'TEC']])

In [ ]:
print(df.iloc[[50, 60, 70]][['DATETIME', 'TEC']])

In [ ]:
df.to_pickle('sjsp271-2024-09-27_selected_0050_to_0110.pkl')

# EMBRACE

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Download TF_EMBRACE_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
df_embrace_maps_2024 = pd.read_pickle('/content/TF_EMBRACE_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl')

In [ ]:
df_embrace_maps_2024

In [ ]:
data = '2024-09-27'
times = ['00:50:00', '01:00:00', '01:10:00']

timestamps = [f'{data} {time}' for time in times]

result = df_embrace_maps_2024.loc[timestamps]

In [ ]:
result

In [ ]:
embrace_maps = np.array(result.iloc[:]['TECMAP'])

In [ ]:
np_embrace_maps = []
for i in range(len(embrace_maps)):
    np_embrace_maps.append(embrace_maps[i])
np_embrace_maps = np.array(np_embrace_maps)

In [ ]:
np_embrace_maps

In [ ]:
np_embrace_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(self, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = self.extent
        lat_step, lon_step = self.lat_step, self.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, self.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, self.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, self.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, self.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points


class Embrace(TecMap):
    def __init__(self,
                 extent: Iterable[float] = (-90, -30, -60, 20),
                 lat_step: float = 2,
                 lon_step: float = 2.5):
        super().__init__(extent, lat_step, lon_step)


embrace_tec = Embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-23.21, lon=-45.51)
print(embrace_points)

In [ ]:
embrace_tec = Embrace(extent=(-90, -30, -60, 20), lat_step=2, lon_step=2.5)
embrace_points = embrace_tec.find_nearest_points(lat=-23.21, lon=-45.51)

In [ ]:
embrace_points

In [ ]:
embrace_tec.tec_map = np_embrace_maps[0]

print("The 4 closest points are:")
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordinates: ({lat}°, {lon}°) - Indexes: [{lat_idx}, {lon_idx}] - TEC value {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nInterpolated TEC value for ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

In [ ]:
embrace_tec.tec_map = np_embrace_maps[1]

print("The 4 closest points are:")
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordinates: ({lat}°, {lon}°) - Indexes: [{lat_idx}, {lon_idx}] - TEC value {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nInterpolated TEC value for ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

In [ ]:
embrace_tec.tec_map = np_embrace_maps[2]

print("The 4 closest points are:")
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = embrace_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordinates: ({lat}°, {lon}°) - Indexes: [{lat_idx}, {lon_idx}] - TEC value {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in embrace_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(embrace_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in embrace_points]
lons = [p[1] for p in embrace_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(embrace_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nInterpolated TEC value for ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

# MAGGIA

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
#Download TF_MAGGIA_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl from https://doi.org/10.5281/zenodo.15453941

In [ ]:
df_maggia_maps_2024 = pd.read_pickle('/content/TF_MAGGIA_TEC_maps_intersection_case_study_from_Sept_05_2024_to_Dec_2024.pkl')

In [ ]:
df_maggia_maps_2024

In [ ]:
df_maggia_maps_2024.set_index('DATETIME', inplace=True)

In [ ]:
df_maggia_maps_2024

In [ ]:
data = '2024-09-27'
times = ['00:50:00', '01:00:00', '01:10:00']

timestamps = [f'{data} {hourrio}' for hourrio in times]

result = df_maggia_maps_2024.loc[timestamps]

In [ ]:
result

In [ ]:
mapas_maggia = np.array(result.iloc[:]['TECMAP'])

In [ ]:
mapas_maggia

In [ ]:
np_maggia_maps = []
for i in range(len(mapas_maggia)):
    np_maggia_maps.append(mapas_maggia[i])
np_maggia_maps = np.array(np_maggia_maps)

In [ ]:
np_maggia_maps

In [ ]:
np_maggia_maps.shape

In [ ]:
import numpy as np
from typing import Iterable

class TecMap:

    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        self.extent = extent
        self.lat_step = lat_step
        self.lon_step = lon_step

        lon_min, lon_max, lat_min, lat_max = extent
        n_lat = int((lat_max - lat_min) / lat_step) + 1
        n_lon = int((lon_max - lon_min) / lon_step) + 1

        self.grid_shape = (n_lat, n_lon)
        self.tec_map = None

    def find_nearest_points(tec_obj, lat: float, lon: float):

        lon_min, lon_max, lat_min, lat_max = tec_obj.extent
        lat_step, lon_step = tec_obj.lat_step, tec_obj.lon_step

        lat_idx_lower = int((lat - lat_min) / lat_step)
        lat_idx_upper = lat_idx_lower + 1

        lat_idx_lower = max(0, min(lat_idx_lower, tec_obj.grid_shape[0] - 1))
        lat_idx_upper = max(0, min(lat_idx_upper, tec_obj.grid_shape[0] - 1))

        lon_idx_lower = int((lon - lon_min) / lon_step)
        lon_idx_upper = lon_idx_lower + 1

        lon_idx_lower = max(0, min(lon_idx_lower, tec_obj.grid_shape[1] - 1))
        lon_idx_upper = max(0, min(lon_idx_upper, tec_obj.grid_shape[1] - 1))

        lat_lower = lat_min + lat_idx_lower * lat_step
        lat_upper = lat_min + lat_idx_upper * lat_step
        lon_lower = lon_min + lon_idx_lower * lon_step
        lon_upper = lon_min + lon_idx_upper * lon_step

        points = []

        if lat_idx_lower != lat_idx_upper and lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower),
                (lat_upper, lon_upper, lat_idx_upper, lon_idx_upper)
            ]
        elif lat_idx_lower != lat_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_upper, lon_lower, lat_idx_upper, lon_idx_lower)
            ]
        elif lon_idx_lower != lon_idx_upper:
            points = [
                (lat_lower, lon_lower, lat_idx_lower, lon_idx_lower),
                (lat_lower, lon_upper, lat_idx_lower, lon_idx_upper)
            ]
        else:
            points = [(lat_lower, lon_lower, lat_idx_lower, lon_idx_lower)]

        return points

class Maggia(TecMap):

    def __init__(self,
                 extent: Iterable[float] = (-110, 0, -80, 40),
                 lat_step: float = 0.5,
                 lon_step: float = 0.5):
        super().__init__(extent, lat_step, lon_step)

In [ ]:
maggia_tec = Maggia(extent=(-110, 0, -80, 40), lat_step=0.5, lon_step=0.5)
maggia_points = maggia_tec.find_nearest_points(lat=-23.21, lon=-45.51)

In [ ]:
maggia_points

In [ ]:
maggia_tec.tec_map = np_maggia_maps[0]

print("The 4 closest points are:")
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordinates: ({lat}°, {lon}°) - Indexes: [{lat_idx}, {lon_idx}] - TEC value {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nInterpolated TEC value for ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

In [ ]:
maggia_tec.tec_map = np_maggia_maps[1]

print("The 4 closest points are:")
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordinates: ({lat}°, {lon}°) - Indexes: [{lat_idx}, {lon_idx}] - TEC value {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nInterpolated TEC value for ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")

In [ ]:
maggia_tec.tec_map = np_maggia_maps[2]

print("The 4 closest points are:")
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point

    tec_value = maggia_tec.tec_map[lat_idx, lon_idx]

    print(f"Coordinates: ({lat}°, {lon}°) - Indexes: [{lat_idx}, {lon_idx}] - TEC value {tec_value}")

target_lat = -23.21
target_lon = -45.51

values = []
for point in maggia_points:
    lat, lon, lat_idx, lon_idx = point
    values.append(maggia_tec.tec_map[lat_idx, lon_idx])

lats = [p[0] for p in maggia_points]
lons = [p[1] for p in maggia_points]

lat_min, lat_max = min(lats), max(lats)
lon_min, lon_max = min(lons), max(lons)

point_values = {}
for i, point in enumerate(maggia_points):
    point_values[(point[0], point[1])] = values[i]

f_00 = point_values[(lat_min, lon_min)]
f_01 = point_values[(lat_min, lon_max)]
f_10 = point_values[(lat_max, lon_min)]
f_11 = point_values[(lat_max, lon_max)]

t_x = (target_lon - lon_min) / (lon_max - lon_min) if lon_max != lon_min else 0
t_y = (target_lat - lat_min) / (lat_max - lat_min) if lat_max != lat_min else 0

interpolated_value = (1 - t_x) * (1 - t_y) * f_00 + \
                     t_x * (1 - t_y) * f_01 + \
                     (1 - t_x) * t_y * f_10 + \
                     t_x * t_y * f_11

print(f"\nInterpolated TEC value for ({target_lat}°, {target_lon}°): {interpolated_value:.4f}")